# RT-DETR-L: Tuned 5-Fold Cross-Validation

## Overview

This notebook trains and evaluates five RT-DETR-L object detection models on the VisDrone dataset. The models are later used for our Weighted Boxes Fusion ensemble.

## Methodology

- **Architecture:** RT-DETR-L, query-based object detection
- **Pretrained weights:** `rtdetr-l.pt`
- **Validation strategy:** 5-fold video-level cross-validation
- **Dataset:** 5,439 processed images across 63 video sequences
- **Training frames:** Every 10th frame
- **Input resolution:** 960px
- **Training:** 8 epochs per fold
- **Optimizer:** AdamW, initial learning rate 0.0002
- **Batch size:** 1
- **Random seed:** 42
- **Evaluation:** COCOeval using `pycocotools`

The validation splits were checked to ensure that training and validation data did not contain frames from the same video.

## Results

| Fold | Validation images | mAP@100 |
|---|---:|---:|
| 0 | 1,211 | 0.19033 |
| 1 | 1,054 | 0.20930 |
| 2 | 1,077 | 0.25266 |
| 3 | 1,078 | 0.24005 |
| 4 | 1,019 | **0.28643** |

**Global out-of-fold evaluation:**

| Metric | Value |
|---|---:|
| mAP@100 | **0.22514** |
| AP50@500 | 0.43044 |
| AP75@500 | 0.20716 |
| AP_small@500 | 0.08208 |

Fold 4 achieved the highest individual validation mAP and was used for our single-model Kaggle submission.

The five trained checkpoints were subsequently combined using Weighted Boxes Fusion in a separate notebook.

## Reproducibility

The experiment was performed on a Kaggle Tesla T4 GPU using `ultralytics==8.4.202`.

Training data and pretrained weights were loaded from the `rtdetr-tuned-training-data` Kaggle dataset.

The notebook saves:

- Five trained checkpoints (`best.pt`)
- Per-fold COCO detection results
- Per-fold evaluation metrics
- Global out-of-fold evaluation results

The pretrained weights and datasets are not included in the GitHub repository.

In [1]:
from pathlib import Path
import zipfile
import shutil
import torch

INPUT = Path("/kaggle/input")

print("=" * 60)
print("RT-DETR-L TUNED — PRE-TRAINING CHECK")
print("=" * 60)

# 1. GPU
print("\n1. GPU CHECK")
gpu_ok = torch.cuda.is_available()
print("GPU available:", gpu_ok)

if gpu_ok:
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

# 2. Kaggle inputs
print("\n2. KAGGLE INPUTS")

for p in sorted(INPUT.iterdir()):
    print(" -", p)

# 3. Pretrained RT-DETR weights
print("\n3. PRETRAINED MODEL")

weights = list(INPUT.rglob("rtdetr-l.pt"))

print("Models found:", len(weights))

for p in weights:
    print("Path:", p)
    print("Size:", round(p.stat().st_size / 1024**2, 2), "MB")

# 4. Dataset
print("\n4. VISDRONE DATASET")

archives = list(INPUT.rglob("data_prepared.zip"))

extracted = [
    p.parent for p in INPUT.rglob("fold0_train.txt")
    if (p.parent / "images").is_dir()
]

data_ok = False

if archives:
    archive = archives[0]

    print("ZIP found:", archive)
    print("ZIP size:", round(archive.stat().st_size / 1024**3, 2), "GB")

    with zipfile.ZipFile(archive) as z:
        names = z.namelist()

        image_count = sum(
            "/images/" in "/" + n and n.endswith(".jpg")
            for n in names
        )

        label_count = sum(
            "/labels/" in "/" + n and n.endswith(".txt")
            for n in names
        )

        required = [
            "fold0_train.txt",
            "fold0_val.txt",
            *[f"coco/fold{i}_val.json" for i in range(5)]
        ]

        missing = [
            r for r in required
            if not any(n.endswith("/" + r) for n in names)
        ]

        data_ok = (
            image_count == 5439
            and label_count == 5439
            and not missing
        )

        print("Images:", image_count)
        print("Labels:", label_count)
        print("Missing required files:", missing)
        print("ZIP structure valid:", data_ok)

elif extracted:
    prep = extracted[0]

    image_count = len(list((prep / "images").glob("*.jpg")))
    label_count = len(list((prep / "labels").glob("*.txt")))

    data_ok = (
        image_count == 5439
        and label_count == 5439
        and all(
            (prep / f"coco/fold{i}_val.json").is_file()
            for i in range(5)
        )
        and all(
            (prep / f"fold{i}_train.txt").is_file()
            and (prep / f"fold{i}_val.txt").is_file()
            for i in range(5)
        )
    )

    print("Extracted dataset:", prep)
    print("Images:", image_count)
    print("Labels:", label_count)
    print("Dataset valid:", data_ok)

else:
    print("DATASET NOT FOUND!")

# 5. Disk space
print("\n5. AVAILABLE DISK SPACE")

free_gb = shutil.disk_usage("/kaggle/working").free / 1024**3
print("Free space:", round(free_gb, 2), "GB")

# 6. Final check
print("\n" + "=" * 60)
print("FINAL STATUS")
print("=" * 60)

checks = {
    "GPU": gpu_ok,
    "Pretrained model": len(weights) == 1,
    "VisDrone dataset": data_ok,
    "Disk space": free_gb >= 6,
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

if all(checks.values()):
    print("\nALL CHECKS PASSED!")
    print("READY FOR RT-DETR-L TUNED TRAINING!")
else:
    print("\nNOT READY — FIX FAILED CHECKS BEFORE TRAINING!")

RT-DETR-L TUNED — PRE-TRAINING CHECK

1. GPU CHECK
GPU available: True
GPU: Tesla T4
GPU memory: 14.56 GB

2. KAGGLE INPUTS
 - /kaggle/input/datasets

3. PRETRAINED MODEL
Models found: 1
Path: /kaggle/input/datasets/vikhimiak/rtdetr-tuned-training-data/rtdetr-l.pt
Size: 63.43 MB

4. VISDRONE DATASET
Extracted dataset: /kaggle/input/datasets/vikhimiak/rtdetr-tuned-training-data/data_prepared/data_prepared
Images: 5439
Labels: 5439
Dataset valid: True

5. AVAILABLE DISK SPACE
Free space: 19.5 GB

FINAL STATUS
GPU: PASS
Pretrained model: PASS
VisDrone dataset: PASS
Disk space: PASS

ALL CHECKS PASSED!
READY FOR RT-DETR-L TUNED TRAINING!


In [3]:
from pathlib import Path
import json

PREP = Path(
    "/kaggle/input/datasets/vikhimiak/"
    "rtdetr-tuned-training-data/data_prepared/data_prepared"
)

all_val = set()
train_counts = []
val_counts = []

def read_names(path):
    return [
        Path(s.strip()).name
        for s in path.read_text().splitlines()
        if s.strip()
    ]

def video_id(name):
    return Path(name).stem.rsplit("_", 1)[0]

print("FINAL 5-FOLD DATA VALIDATION")
print("=" * 60)

for fold in range(5):
    train = read_names(PREP / f"fold{fold}_train.txt")
    val = read_names(PREP / f"fold{fold}_val.txt")

    selected = [
        name for name in train
        if (
            int(Path(name).stem.rsplit("_", 1)[-1]) - 1
        ) % 10 == 0
    ]

    with open(PREP / f"coco/fold{fold}_val.json") as f:
        coco = json.load(f)

    coco_names = {
        Path(img["file_name"]).name
        for img in coco["images"]
    }

    assert len(train) == len(set(train))
    assert len(val) == len(set(val))
    assert set(train).isdisjoint(set(val))
    assert {video_id(n) for n in train}.isdisjoint(
        {video_id(n) for n in val}
    ), f"Video leakage in Fold {fold}"

    assert len(coco["images"]) == len(val)
    assert coco_names == set(val)
    assert all_val.isdisjoint(set(val)), (
        f"Overlapping validation images in Fold {fold}"
    )

    all_val.update(val)
    train_counts.append(len(selected))
    val_counts.append(len(val))

    print(
        f"Fold {fold}: PASS | "
        f"Train {len(selected)} | Val {len(val)}"
    )

all_images = {
    p.name for p in (PREP / "images").glob("*.jpg")
}

assert all_val == all_images
assert len(all_val) == 5439

print("\nTrain counts:", train_counts)
print("Val counts:", val_counts)
print("Unique validation images:", len(all_val))
print("\nALL SPLIT VALIDATION CHECKS PASSED!")

FINAL 5-FOLD DATA VALIDATION
Fold 0: PASS | Train 2127 | Val 1211
Fold 1: PASS | Train 2203 | Val 1054
Fold 2: PASS | Train 2192 | Val 1077
Fold 3: PASS | Train 2193 | Val 1078
Fold 4: PASS | Train 2221 | Val 1019

Train counts: [2127, 2203, 2192, 2193, 2221]
Val counts: [1211, 1054, 1077, 1078, 1019]
Unique validation images: 5439

ALL SPLIT VALIDATION CHECKS PASSED!


In [4]:
# ============================================================
# RT-DETR-L TUNED — COMPLETE 5-FOLD CV ON KAGGLE
# 8 epochs per fold | imgsz=960 | AdamW | Tesla T4
# ============================================================

import sys
import subprocess
import os
import json
import time
import gc
import zipfile
from pathlib import Path

# Install fixed versions
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.4.202", "pycocotools"
])

import yaml
import torch
import pandas as pd
import numpy as np

from ultralytics import RTDETR
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ============================================================
# 1. ENVIRONMENT AND FILES
# ============================================================

assert torch.cuda.is_available(), "Enable GPU in Kaggle settings!"

print("GPU:", torch.cuda.get_device_name(0))

INPUT = Path("/kaggle/input")
OUTPUT = Path("/kaggle/working/rtdetr_tuned")
OUTPUT.mkdir(parents=True, exist_ok=True)

RUNS = OUTPUT / "cv_runs"
CONFIGS = OUTPUT / "configs"
RUNS.mkdir(exist_ok=True)
CONFIGS.mkdir(exist_ok=True)

# Find pretrained weights
weights_found = list(INPUT.rglob("rtdetr-l.pt"))

assert len(weights_found) == 1, (
    "Upload rtdetr-l.pt in the Kaggle training Dataset."
)

WEIGHTS = weights_found[0]

# Locate dataset archive or existing extracted dataset
zips = list(INPUT.rglob("data_prepared.zip"))

if zips:
    DATA_PARENT = Path("/kaggle/working/hw2")
    DATA_PARENT.mkdir(parents=True, exist_ok=True)
    PREP = DATA_PARENT / "data_prepared"

    if not (PREP / "coco/fold4_val.json").is_file():
        print("Extracting VisDrone dataset...")

        with zipfile.ZipFile(zips[0]) as z:
            for item in z.infolist():
                dest = (DATA_PARENT / item.filename).resolve()
                assert dest.is_relative_to(DATA_PARENT.resolve())
            z.extractall(DATA_PARENT)

else:
    prepared = [
        p.parent for p in INPUT.rglob("fold0_train.txt")
        if (p.parent / "images").is_dir()
        and (p.parent / "coco/fold0_val.json").is_file()
    ]

    assert len(prepared) == 1, (
        "Could not find data_prepared.zip or extracted data."
    )
    PREP = prepared[0]

assert (PREP / "images").is_dir()
assert (PREP / "labels").is_dir()
assert len(list((PREP / "images").glob("*.jpg"))) == 5439
assert len(list((PREP / "labels").glob("*.txt"))) == 5439

print("Dataset:", PREP)
print("Pretrained weights:", WEIGHTS)

# ============================================================
# 2. TRAINING PARAMETERS — SAME AS COLAB TUNED
# ============================================================

CLASSES = [
    "person", "bicycle", "car", "van",
    "truck", "tricycle", "bus", "motor"
]

EXPECTED_TRAIN = [2127, 2203, 2192, 2193, 2221]
EXPECTED_VAL = [1211, 1054, 1077, 1078, 1019]

TRAIN_ARGS = {
    "epochs": 8,
    "imgsz": 960,
    "batch": 1,
    "nbs": 8,
    "optimizer": "AdamW",
    "lr0": 0.0002,
    "warmup_epochs": 1.0,
    "warmup_bias_lr": 0.0,
    "scale": 0.5,
    "device": 0,
    "workers": 2,
    "seed": 42,
    "deterministic": False,
    "amp": True,
    "val": True,
    "save": True,
    "save_period": -1,  # Keep best.pt / last.pt; avoid extra disk use
    "plots": False,
    "cache": False,
    "exist_ok": True,
}

with open(OUTPUT / "training_config.json", "w") as f:
    json.dump(TRAIN_ARGS, f, indent=2)

# ============================================================
# 3. HELPERS
# ============================================================

def save_json(path, data):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w") as f:
        json.dump(data, f)
    os.replace(tmp, path)


def make_data_yaml(fold):

    def read_names(filename):
        return [
            Path(s.strip()).name
            for s in filename.read_text().splitlines()
            if s.strip()
        ]

    train_names = read_names(PREP / f"fold{fold}_train.txt")
    val_names = read_names(PREP / f"fold{fold}_val.txt")

    # Same 1-in-10 training frame selection as Colab
    train_selected = [
        name for name in train_names
        if (
            int(Path(name).stem.rsplit("_", 1)[-1]) - 1
        ) % 10 == 0
    ]

    assert len(train_selected) == EXPECTED_TRAIN[fold]
    assert len(val_names) == EXPECTED_VAL[fold]

    def video_id(name):
        return Path(name).stem.rsplit("_", 1)[0]

    assert {
        video_id(n) for n in train_names
    }.isdisjoint({
        video_id(n) for n in val_names
    }), "Video leakage detected!"

    with open(PREP / f"coco/fold{fold}_val.json") as f:
        coco_info = json.load(f)

    assert {
        Path(x["file_name"]).name
        for x in coco_info["images"]
    } == set(val_names)

    train_txt = CONFIGS / f"fold{fold}_train.txt"
    val_txt = CONFIGS / f"fold{fold}_val.txt"

    train_paths = [
        str(PREP / "images" / name)
        for name in train_selected
    ]

    val_paths = [
        str(PREP / "images" / name)
        for name in val_names
    ]

    assert all(Path(p).is_file() for p in train_paths + val_paths)

    train_txt.write_text("\n".join(train_paths) + "\n")
    val_txt.write_text("\n".join(val_paths) + "\n")

    config = CONFIGS / f"fold{fold}.yaml"

    with open(config, "w") as f:
        yaml.safe_dump({
            "path": str(PREP),
            "train": str(train_txt),
            "val": str(val_txt),
            "nc": 8,
            "names": CLASSES
        }, f, sort_keys=False)

    return config


def get_ap(evaluator, area=0, maxdet=2, iou=None):
    p = evaluator.eval["precision"]

    if iou is not None:
        p = p[iou:iou + 1]

    p = p[:, :, :, area, maxdet]
    valid = p[p > -1]

    return float(valid.mean()) if valid.size else None


def evaluate_predictions(gt_path, detections):
    coco_gt = COCO(str(gt_path))
    coco_dt = coco_gt.loadRes(detections)

    evaluator = COCOeval(coco_gt, coco_dt, "bbox")
    evaluator.params.imgIds = sorted(coco_gt.getImgIds())
    evaluator.params.maxDets = [1, 100, 500]

    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    metrics = {
        "mAP_maxDets100": get_ap(evaluator, 0, 1),
        "mAP_maxDets500": get_ap(evaluator, 0, 2),
        "AP50_maxDets500": get_ap(evaluator, 0, 2, 0),
        "AP75_maxDets500": get_ap(evaluator, 0, 2, 5),
        "AP_small": get_ap(evaluator, 1, 2),
        "AP_medium": get_ap(evaluator, 2, 2),
        "AP_large": get_ap(evaluator, 3, 2)
    }

    del evaluator, coco_dt, coco_gt
    gc.collect()

    return metrics


# ============================================================
# 4. TRAIN AND EVALUATE ALL FIVE FOLDS
# ============================================================

total_start = time.perf_counter()
fold_results = []

for fold in range(5):

    print("\n" + "=" * 70)
    print(f"RT-DETR-L TUNED — FOLD {fold}/4")
    print("=" * 70)

    yaml_path = make_data_yaml(fold)

    run_dir = RUNS / f"tuned_cv_fold{fold}"
    best_pt = run_dir / "weights/best.pt"
    last_pt = run_dir / "weights/last.pt"
    results_csv = run_dir / "results.csv"

    metrics_path = OUTPUT / f"tuned_fold{fold}_metrics.json"
    detections_path = OUTPUT / f"tuned_fold{fold}_val_dets.json"

    # TRAIN
    completed = 0

    if results_csv.is_file():
        completed = len(pd.read_csv(results_csv))

    if completed == 8 and best_pt.is_file():

        print("Training already completed — skipping.")

    elif completed > 0 and last_pt.is_file():

        print(f"Resuming Fold {fold} from epoch {completed + 1}...")

        model = RTDETR(str(last_pt))
        model.train(
            resume=True,
            data=str(yaml_path),
            device=0,
            workers=2
        )

        del model

    else:

        print(f"Starting Fold {fold} from pretrained RT-DETR-L")

        model = RTDETR(str(WEIGHTS))

        model.train(
            data=str(yaml_path),
            project=str(RUNS),
            name=f"tuned_cv_fold{fold}",
            **TRAIN_ARGS
        )

        del model

    gc.collect()
    torch.cuda.empty_cache()

    assert best_pt.is_file()
    assert len(pd.read_csv(results_csv)) == 8

    print(f"FOLD {fold} TRAINING COMPLETE")

    # COCO INFERENCE
    if metrics_path.is_file() and detections_path.is_file():

        print("Existing COCOeval results found — skipping.")

        with open(metrics_path) as f:
            metrics = json.load(f)

    else:

        print(f"Starting Fold {fold} COCO inference...")

        gt_path = PREP / f"coco/fold{fold}_val.json"

        with open(gt_path) as f:
            ground_truth = json.load(f)

        images = sorted(
            ground_truth["images"],
            key=lambda x: str(x["id"])
        )

        assert len(images) == EXPECTED_VAL[fold]

        model = RTDETR(str(best_pt))
        detections = []

        for idx, img in enumerate(images):

            image_path = PREP / img["file_name"]
            assert image_path.is_file(), image_path

            result = model.predict(
                source=str(image_path),
                imgsz=960,
                conf=0.01,
                iou=0.7,
                max_det=500,
                device=0,
                verbose=False
            )[0]

            height, width = result.orig_shape

            assert width == img["width"]
            assert height == img["height"]

            for box in result.boxes:

                cls = int(box.cls.item())
                score = float(box.conf.item())

                assert 0 <= cls < 8

                x1, y1, x2, y2 = box.xyxy[0].tolist()

                x1 = max(0.0, min(float(x1), width))
                x2 = max(0.0, min(float(x2), width))
                y1 = max(0.0, min(float(y1), height))
                y2 = max(0.0, min(float(y2), height))

                w = x2 - x1
                h = y2 - y1

                if w <= 0 or h <= 0:
                    continue

                detections.append({
                    "image_id": img["id"],
                    "category_id": cls + 1,
                    "bbox": [x1, y1, w, h],
                    "score": score
                })

            if (idx + 1) % 200 == 0:
                print(
                    f"Fold {fold} inference: "
                    f"{idx + 1}/{len(images)}"
                )

        save_json(detections_path, detections)

        del model
        gc.collect()
        torch.cuda.empty_cache()

        metrics = evaluate_predictions(gt_path, detections)

        metrics.update({
            "fold": fold,
            "epochs": 8,
            "validation_images": len(images),
            "detections": len(detections)
        })

        save_json(metrics_path, metrics)

        del detections, ground_truth
        gc.collect()

    fold_results.append(metrics)

    print(f"\nFOLD {fold} COCO mAP@100: "
          f"{metrics['mAP_maxDets100']:.6f}")

    print(
        "Total elapsed:",
        round((time.perf_counter() - total_start) / 3600, 2),
        "hours"
    )

# ============================================================
# 5. FIVE-FOLD SUMMARY
# ============================================================

summary = pd.DataFrame(fold_results)
summary.to_csv(OUTPUT / "tuned_5fold_summary.csv", index=False)

print("\n" + "=" * 70)
print("RT-DETR-L TUNED — FIVE-FOLD RESULTS")
print("=" * 70)

display(summary.round(5))

assert summary["validation_images"].sum() == 5439

# ============================================================
# 6. GLOBAL OUT-OF-FOLD COCO EVALUATION
# ============================================================

print("\nBUILDING GLOBAL OUT-OF-FOLD EVALUATION")

oof_gt = {
    "info": {"description": "RT-DETR-L Tuned 5-fold OOF"},
    "licenses": [],
    "images": [],
    "annotations": [],
    "categories": []
}

oof_detections = []
next_image_id = 1
next_annotation_id = 1
seen_images = set()
category_signature = None

for fold in range(5):

    with open(PREP / f"coco/fold{fold}_val.json") as f:
        gt = json.load(f)

    with open(OUTPUT / f"tuned_fold{fold}_val_dets.json") as f:
        dets = json.load(f)

    signature = sorted(
        (c["id"], c["name"]) for c in gt["categories"]
    )

    if category_signature is None:
        category_signature = signature
        oof_gt["categories"] = gt["categories"]
    else:
        assert signature == category_signature

    id_map = {}

    for img in gt["images"]:

        old_id = img["id"]
        image_name = Path(img["file_name"]).name

        assert image_name not in seen_images
        seen_images.add(image_name)

        id_map[old_id] = next_image_id

        new_img = img.copy()
        new_img["id"] = next_image_id

        oof_gt["images"].append(new_img)
        next_image_id += 1

    for ann in gt["annotations"]:

        new_ann = ann.copy()
        new_ann["image_id"] = id_map[ann["image_id"]]
        new_ann["id"] = next_annotation_id

        oof_gt["annotations"].append(new_ann)
        next_annotation_id += 1

    for det in dets:

        new_det = det.copy()
        new_det["image_id"] = id_map[det["image_id"]]

        oof_detections.append(new_det)

    print(f"Fold {fold}: merged {len(gt['images'])} images")

    del gt, dets
    gc.collect()

assert len(oof_gt["images"]) == 5439
assert len(seen_images) == 5439

OOF_GT_PATH = OUTPUT / "tuned_oof_gt.json"
OOF_DETS_PATH = OUTPUT / "tuned_oof_dets.json"
OOF_METRICS_PATH = OUTPUT / "tuned_oof_metrics.json"

save_json(OOF_GT_PATH, oof_gt)
save_json(OOF_DETS_PATH, oof_detections)

del oof_gt
gc.collect()

print("\nSTARTING GLOBAL OOF COCOeval...")

oof_metrics = evaluate_predictions(
    OOF_GT_PATH,
    oof_detections
)

oof_metrics.update({
    "model": "RT-DETR-L Tuned",
    "folds": 5,
    "epochs_per_fold": 8,
    "images": 5439,
    "detections": len(oof_detections)
})

save_json(OOF_METRICS_PATH, oof_metrics)

# ============================================================
# 7. FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("RT-DETR-L TUNED — FINAL OOF RESULTS")
print("=" * 70)

for key, value in oof_metrics.items():
    print(f"{key}: {value}")

print("\nCOMPARISON")

print("RT-DETR-L Baseline OOF:", 0.0508267444)
print("RT-DETR-L Tuned OOF:",
      round(oof_metrics["mAP_maxDets100"], 6))
print("YOLO11n CV OOF:", 0.148355)

print(
    "\nTotal experiment time:",
    round((time.perf_counter() - total_start) / 3600, 2),
    "hours"
)

print("\nResults saved to:", OUTPUT)
print("ALL FIVE TUNED FOLDS COMPLETED SUCCESSFULLY!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
GPU: Tesla T4
Dataset: /kaggle/input/datasets/vikhimiak/rtdetr-tuned-training-data/data_prepared/data_prepared
Pretrained weights: /kaggle/input/datasets/vikhimiak/rtdetr-tuned-training-data/rtdetr-l.pt

RT-DETR-L TUNED — FOLD 0/4
Starting Fold 0 from pretrained RT-DETR-L
New https://pypi.org/project/ultralytics/8.4.212 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


        2/8      2.24G     0.6816     0.4914    0.09169          8        960: 100% ━━━━━━━━━━━━ 2127/2127 4.7it/s 7:33
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 606/606 10.5it/s 57.7s
                   all       1211      54458      0.449      0.352      0.333      0.164

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        3/8      2.24G     0.6376     0.4773    0.08306          5        960: 100% ━━━━━━━━━━━━ 2127/2127 4.7it/s 7:32
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 606/606 10.5it/s 57.8s
                   all       1211      54458      0.485      0.354      0.364      0.186

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        4/8      2.24G     0.6153      0.466    0.07758         42        960: 100% ━━━━━━━━━━━━ 2127/2127 4.7it/s 7:30
                 Class     Images  Instanc

,mAP_maxDets100,mAP_maxDets500,AP50_maxDets500,AP75_maxDets500,AP_small,AP_medium,AP_large,fold,epochs,validation_images,detections
0,0.19033,0.19069,0.36890,0.17024,0.06625,0.22290,0.45125,0,8,1211,362697
1,0.20930,0.21066,0.42041,0.18422,0.06641,0.18762,0.46150,1,8,1054,316200
2,0.25266,0.25274,0.46259,0.24750,0.14187,0.26856,0.38287,2,8,1077,323100
3,0.24005,0.24014,0.45381,0.22078,0.14699,0.19854,0.42358,3,8,1078,323400
4,0.28643,0.28657,0.50295,0.28224,0.11108,0.28777,0.46239,4,8,1019,305700



BUILDING GLOBAL OUT-OF-FOLD EVALUATION
Fold 0: merged 1211 images
Fold 1: merged 1054 images
Fold 2: merged 1077 images
Fold 3: merged 1078 images
Fold 4: merged 1019 images

STARTING GLOBAL OOF COCOeval...
loading annotations into memory...
Done (t=1.24s)
creating index...
index created!
Loading and preparing results...
DONE (t=5.50s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=427.98s).
Accumulating evaluation results...
DONE (t=28.58s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.225
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=500 ] = 0.430
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=500 ] = 0.207
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=500 ] = 0.082
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=500 ] = 0.224
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=500 ] = 0.403
 Average Recal

In [5]:
from pathlib import Path
import json
import shutil

ROOT = Path("/kaggle/working/rtdetr_tuned")

print("RT-DETR-L TUNED — FINAL FILE CHECK")
print("=" * 60)

for fold in range(5):
    best = ROOT / f"cv_runs/tuned_cv_fold{fold}/weights/best.pt"
    metrics = ROOT / f"tuned_fold{fold}_metrics.json"
    predictions = ROOT / f"tuned_fold{fold}_val_dets.json"

    assert best.is_file(), best
    assert metrics.is_file(), metrics
    assert predictions.is_file(), predictions

    with open(metrics) as f:
        result = json.load(f)

    print(
        f"Fold {fold}: PASS | "
        f"mAP@100={result['mAP_maxDets100']:.6f}"
    )

oof_file = ROOT / "tuned_oof_metrics.json"
assert oof_file.is_file()

with open(oof_file) as f:
    oof = json.load(f)

print("\nGlobal OOF mAP@100:", oof["mAP_maxDets100"])

# Prepare Fold 4 for Kaggle inference
source = ROOT / "cv_runs/tuned_cv_fold4/weights/best.pt"
target = Path("/kaggle/working/rtdetr_tuned_fold4_best.pt")

shutil.copy2(source, target)

assert target.is_file()
assert target.stat().st_size == source.stat().st_size

print("\nKaggle submission model:", target)
print("Model size:", round(target.stat().st_size / 1024**2, 2), "MB")

print("\nALL CHECKS PASSED — READY TO SAVE!")

RT-DETR-L TUNED — FINAL FILE CHECK
Fold 0: PASS | mAP@100=0.190330
Fold 1: PASS | mAP@100=0.209298
Fold 2: PASS | mAP@100=0.252660
Fold 3: PASS | mAP@100=0.240050
Fold 4: PASS | mAP@100=0.286432

Global OOF mAP@100: 0.2251376440608354

Kaggle submission model: /kaggle/working/rtdetr_tuned_fold4_best.pt
Model size: 63.26 MB

ALL CHECKS PASSED — READY TO SAVE!
